# Dimensionless QHO1D finite-difference spectrum

**Prerequisite:** [Finite-difference quantum harmonic oscillator](../../../docs/lecture-notes/qm/qho1d/qho1d__finite_difference.md)

This laboratory preserves the exploratory graph of shifted low-energy wavefunctions over the harmonic potential while moving reusable grid, Hamiltonian, normalization, and eigensolver behavior into the package.

## Learning objectives

1. Construct a bounded homogeneous-Dirichlet representation of an unbounded oscillator.
2. Compare low represented energies with $E_n=\hbar\omega(n+1/2)$.
3. Check continuous normalization of finite-difference eigenvectors.
4. Visualize shifted eigenfunctions together with the potential.

All parameters and coordinates are dimensionless illustrative values. The finite interval is a numerical truncation, not a physical hard-wall oscillator model.


## Represented Hamiltonian

The continuum operator is

$$
\hat H=-\frac{\hbar^2}{2m}\frac{d^2}{dx^2}
+\frac12m\omega^2x^2.
$$

The second derivative uses a centered three-point stencil on interior points. The computational endpoints impose homogeneous Dirichlet values sufficiently far from the low-state support. Both grid spacing and interval truncation contribute numerical error.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.qm.qho1d.tise.finite_difference.model import (
    DimensionlessQho1D,
)
from projectkoios.physkit.units import ScalarQuantity, Unitless

In [ ]:
def unitless_scalar(magnitude: float) -> ScalarQuantity:
    return ScalarQuantity(magnitude=magnitude, unit=Unitless())


model = DimensionlessQho1D(
    mass=unitless_scalar(1.0),
    angular_frequency=unitless_scalar(1.0),
    hbar=unitless_scalar(1.0),
)
solution = model.solve_tise_finite_difference(
    lower_bound=unitless_scalar(-8.0),
    upper_bound=unitless_scalar(8.0),
    point_count=400,
)
comparison_count = 8
analytical_energies = model.analytical_energies(count=comparison_count).magnitude
numerical_energies = solution.energies.magnitude[:comparison_count]
relative_energy_errors = np.abs(
    (numerical_energies - analytical_energies) / analytical_energies
)

In [ ]:
represented_norms = (
    np.sum(
        np.abs(solution.eigenfunctions.magnitude[:, :comparison_count]) ** 2,
        axis=0,
    )
    * solution.grid_spacing.magnitude
)
assert np.allclose(represented_norms, 1.0, rtol=0.0, atol=2.0e-14)
assert np.all(relative_energy_errors < 1.8e-3)
assert np.allclose(
    solution.hamiltonian.magnitude,
    solution.hamiltonian.magnitude.T,
    rtol=0.0,
    atol=0.0,
)
numerical_energies, analytical_energies, relative_energy_errors

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(13.0, 5.0))
shown_state_count = 4
for state_index in range(shown_state_count):
    eigenfunction = solution.eigenfunctions.magnitude[:, state_index].copy()
    center_index = int(np.argmin(np.abs(solution.coordinates.magnitude)))
    if eigenfunction[center_index] < 0.0:
        eigenfunction *= -1.0
    axes[0].plot(
        solution.coordinates.magnitude,
        eigenfunction + numerical_energies[state_index],
        label=rf"$n={state_index}$",
    )
axes[0].plot(
    solution.coordinates.magnitude,
    solution.potential_energy.magnitude,
    color="black",
    linestyle="--",
    label=r"$V(x)=x^2/2$",
)
axes[0].set(
    xlim=(-5.0, 5.0),
    ylim=(0.0, 6.0),
    xlabel="reduced coordinate $x$",
    ylabel="energy / shifted eigenfunction",
    title="Low states over the harmonic potential",
)
axes[0].grid(True)
axes[0].legend()

quantum_numbers = np.arange(comparison_count)
axes[1].semilogy(
    quantum_numbers,
    relative_energy_errors,
    marker="o",
)
axes[1].set(
    xlabel="quantum number $n$",
    ylabel="relative energy error",
    title="Finite-representation energy error",
    xticks=quantum_numbers,
)
axes[1].grid(True)

figure.tight_layout()
plt.show()

## Interpretation

The first panel preserves the exploratory shifted-wavefunction presentation: each continuously normalized eigenfunction is displaced vertically by its represented energy and overlaid on the potential. The second panel makes the approximation error explicit rather than relying on printed energies alone.

Agreement for these low states is numerical verification under the selected interval and grid. It does not establish convergence for every state. The separate root-level Hermite-basis notebook represents the operator in a truncated function basis and therefore has a different error source; it is not a duplicate of this grid calculation.

## Exercises

1. Repeat with several point counts while holding the interval fixed.
2. Repeat with several interval bounds while holding the spacing approximately fixed.
3. Verify the expected even/odd parity of successive eigenfunctions.
4. Compare grid-discretization error with Hermite-basis truncation error without combining them into one metric.
